### Step 1 — Mount Drive and define paths

In [12]:
# Step 1: Mount Google Drive and define paths

from google.colab import drive
from pathlib import Path
import json
import random

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/rectified_flow_ct2dose")

RAW_ROOT_CANDIDATES = [
    BASE_DIR / "data" / "raw" / "46_53-32_cube" / "output",
    BASE_DIR / "data" / "46_53-32_cube",
]

RAW_ROOT = None
for p in RAW_ROOT_CANDIDATES:
    if p.exists():
        RAW_ROOT = p
        break

if RAW_ROOT is None:
    raise FileNotFoundError("Could not find RAW_ROOT. Please check the raw data folder.")

SPLIT_DIR = BASE_DIR / "data" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_ROOT:", RAW_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/rectified_flow_ct2dose
RAW_ROOT: /content/drive/MyDrive/rectified_flow_ct2dose/data/raw/46_53-32_cube/output
SPLIT_DIR: /content/drive/MyDrive/rectified_flow_ct2dose/data/splits


### Step 2 — List all case folders

In [13]:
# Step 2: List all case folders

case_dirs = sorted([p for p in RAW_ROOT.iterdir() if p.is_dir()])

print("Number of case directories:", len(case_dirs))
print("\nAll case IDs:")
for i, p in enumerate(case_dirs):
    print(i, p.name)

Number of case directories: 10

All case IDs:
0 132102389758881090179672567372987664560
1 156339165372145091992366212553627308160
2 169178663511657097619787299759624979616
3 226808136684964871480153898846429021344
4 235101017859661465075472232303048949736
5 277818358016850894820204297874963333096
6 40001034793533568458224025980056277120
7 40838398986272789027252987369168472240
8 62242424098194756750821789925294112700
9 73631120845087000785006921692554887100


### Step 3 — Define train / val / hold-out test cases

In [14]:
# Step 3: Define train / val / hold-out test cases

dev_case_dirs = case_dirs[:8]
holdout_test_case_dirs = case_dirs[8:]

train_case_dirs = dev_case_dirs[:6]
val_case_dirs = dev_case_dirs[6:8]

print("Train cases:", len(train_case_dirs))
for p in train_case_dirs:
    print("  TRAIN:", p.name)

print("\nValidation cases:", len(val_case_dirs))
for p in val_case_dirs:
    print("  VAL:", p.name)

print("\nHold-out test cases:", len(holdout_test_case_dirs))
for p in holdout_test_case_dirs:
    print("  TEST:", p.name)

Train cases: 6
  TRAIN: 132102389758881090179672567372987664560
  TRAIN: 156339165372145091992366212553627308160
  TRAIN: 169178663511657097619787299759624979616
  TRAIN: 226808136684964871480153898846429021344
  TRAIN: 235101017859661465075472232303048949736
  TRAIN: 277818358016850894820204297874963333096

Validation cases: 2
  VAL: 40001034793533568458224025980056277120
  VAL: 40838398986272789027252987369168472240

Hold-out test cases: 2
  TEST: 62242424098194756750821789925294112700
  TEST: 73631120845087000785006921692554887100


### Step 4 — Build a 3D manifest from case directories

In [15]:
# Step 4: Build a 3D manifest from case directories

def build_3d_manifest(case_dirs):
    """
    Build a 3D paired manifest from a list of case directories.

    Each record contains:
    - case_id
    - input_path
    - output_path
    """
    records = []

    for case_dir in case_dirs:
        input_dir = case_dir / "input_cubes"
        output_dir = case_dir / "output_cubes"

        input_files = sorted(input_dir.glob("*.npy"))

        for input_path in input_files:
            output_path = output_dir / input_path.name

            if output_path.exists():
                records.append({
                    "case_id": case_dir.name,
                    "input_path": str(input_path),
                    "output_path": str(output_path),
                })

    return records

### Step 5 — Build full train / val / hold-out test manifests

In [16]:
# Step 5: Build full manifests

train_records_full = build_3d_manifest(train_case_dirs)
val_records_full = build_3d_manifest(val_case_dirs)
holdout_test_records_full = build_3d_manifest(holdout_test_case_dirs)

print("Full train records:", len(train_records_full))
print("Full val records:", len(val_records_full))
print("Full hold-out test records:", len(holdout_test_records_full))

Full train records: 12000
Full val records: 4000
Full hold-out test records: 4000


### Step 6 — Sample 2000 train and 400 val

In [17]:
# Step 6: Sample 2000 train and 500 validation records

def sample_records(records, n_samples, seed=42):
    """
    Randomly sample records with a fixed seed.
    """
    random.seed(seed)
    if len(records) <= n_samples:
        return records
    return random.sample(records, n_samples)

train_records_2000 = sample_records(train_records_full, 2000, seed=42)
val_records_500 = sample_records(val_records_full, 500, seed=42)

print("Sampled train records:", len(train_records_2000))
print("Sampled val records:", len(val_records_500))

Sampled train records: 2000
Sampled val records: 500


### Step 7 — Save the new manifests

In [18]:
# Step 7: Save new manifests

TRAIN_MANIFEST_PHASE3 = SPLIT_DIR / "train_pairs_3d_train_phase3_2000.json"
VAL_MANIFEST_PHASE3 = SPLIT_DIR / "train_pairs_3d_val_phase3_500.json"
HOLDOUT_TEST_MANIFEST_PHASE3 = SPLIT_DIR / "test_pairs_3d_holdout_phase3_full.json"

with open(TRAIN_MANIFEST_PHASE3, "w", encoding="utf-8") as f:
    json.dump(train_records_2000, f, indent=2)

with open(VAL_MANIFEST_PHASE3, "w", encoding="utf-8") as f:
    json.dump(val_records_500, f, indent=2)

with open(HOLDOUT_TEST_MANIFEST_PHASE3, "w", encoding="utf-8") as f:
    json.dump(holdout_test_records_full, f, indent=2)

print("Saved:", TRAIN_MANIFEST_PHASE3)
print("Saved:", VAL_MANIFEST_PHASE3)
print("Saved:", HOLDOUT_TEST_MANIFEST_PHASE3)

Saved: /content/drive/MyDrive/rectified_flow_ct2dose/data/splits/train_pairs_3d_train_phase3_2000.json
Saved: /content/drive/MyDrive/rectified_flow_ct2dose/data/splits/train_pairs_3d_val_phase3_500.json
Saved: /content/drive/MyDrive/rectified_flow_ct2dose/data/splits/test_pairs_3d_holdout_phase3_full.json


### Step 8 — Print a final summary of the split

In [19]:
# Step 8: Print final split summary

print("=== Phase 3 split summary ===")
print("Train cases:", len(train_case_dirs))
print("Validation cases:", len(val_case_dirs))
print("Hold-out test cases:", len(holdout_test_case_dirs))

print("Train samples used:", len(train_records_2000))
print("Validation samples used:", len(val_records_500))
print("Hold-out test samples saved:", len(holdout_test_records_full))

=== Phase 3 split summary ===
Train cases: 6
Validation cases: 2
Hold-out test cases: 2
Train samples used: 2000
Validation samples used: 500
Hold-out test samples saved: 4000
